# Cooperation for compounds from other members: consumption overlap

In [1]:
import pandas as pd
import json
import seaborn as sns
import numpy as np

import networkx as nx

from pyvis.network import Network

In [2]:
media_db = pd.read_csv("https://raw.githubusercontent.com/cdanielmachado/carveme/refs/heads/master/carveme/data/input/media_db.tsv",sep="\t")

In [3]:
with open("../data/Venturelli2018/generated/metid2metname.json") as json_file:
    metid2metname = json.load(json_file)

In [4]:
communities_12_exp_abundance = pd.read_csv("../data/Venturelli2018/generated/communities_12_exp_abundance.tsv",sep="\t",index_col=[0,1])

In [5]:
m9_mets = list(media_db[media_db.medium=="M9[-O2]"].compound.values)
m9_mets.remove("so4")
m9_mets.remove("nh4")
m9_mets.remove("glc__D")

m9_mets = ["M_"+met+"__91__e__93__" for met in m9_mets]

In [6]:
results = pd.read_csv("../output/Venturelli2018/results_steadiercom/NONE_abundance.tsv",sep="\t")

results["mass_rate*frequency"] = results["frequency"]*results["mass_rate"]

In [7]:
results = results[~results.compound.isin(m9_mets)].copy()

In [8]:
def centrality_metrics(network_data,cutoff=0,SP_centrality=True):
    """
    SP_centrality: centrality meassures based on shortest path
    """
    # Betweenness and closeness
    G = nx.Graph()
    G.add_nodes_from(list(set(list(network_data.index.get_level_values(0).unique())+list(network_data.index.get_level_values(1).unique()))))
    
    if SP_centrality:
        edges = [(index[0],index[1],{"weight":1/data} ) for index,data in network_data.items() if data>cutoff]
    
    else:
        edges = [(index[0],index[1],{"weight":data} ) for index,data in network_data.items() if data>cutoff]
        
    G.add_edges_from(edges)
    
    if SP_centrality:
        return pd.DataFrame({"betweeness (1/data)":nx.centrality.betweenness_centrality(G,weight="weight"),
                     "closeness (1/data)":nx.centrality.closeness_centrality(G,distance="weight")})
    else:
        return pd.DataFrame({"degree (data)":dict(G.degree(weight="weight")),
                 "eigenvector (data)":nx.centrality.eigenvector_centrality(G,weight="weight")})


In [9]:
## From https://stackoverflow.com/questions/25668828/how-to-create-colour-gradient-in-python
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np

def colorFader(c1,c2,mix=0): #fade (linear interpolate) from color c1 (at mix=0) to c2 (mix=1)
    c1=np.array(mpl.colors.to_rgb(c1))
    c2=np.array(mpl.colors.to_rgb(c2))
    return mpl.colors.to_hex((1-mix)*c1 + mix*c2)


In [10]:
b1='#f8cdcf' #bright pink
b2='#fa4f56' #red

c1='#ffffff' #white
c2='#0b5394' #blue

In [11]:
def frequency_network(network_data,closeness_centrality_met,cutoff=0):
    net = Network(directed=True)

    nodes = list(set(list(network_data.index.get_level_values(0).unique())+list(network_data.index.get_level_values(1).unique())))
    color = [colorFader(b1,b2,closeness_centrality_met[node]) for node in nodes]
    value = [(closeness_centrality_met[node]*10)**2 for node in nodes]
    net.add_nodes(nodes,color=color,value=value)
    
    
    maximum_corr = network_data.max()
    for index,value in network_data.items():
        if value>cutoff:
            net.add_edge(index[0],index[1],width=(value**2)*10,color=colorFader(c1,c2,value/maximum_corr)) 

    net.barnes_hut(gravity=-1000, central_gravity=0.1, spring_length=150, spring_strength=0.0015, damping=0.09, overlap=0.8)


    template = net.templateEnv.get_template(net.path)
    net.template = template
    return net

### Average frequency

In [12]:
def frequency_network(network_data,closeness_centrality_met,cutoff=0):
    net = Network(directed=True)

    nodes = list(set(list(network_data.index.get_level_values(0).unique())+list(network_data.index.get_level_values(1).unique())))
    color = [colorFader(b1,b2,closeness_centrality_met[node]) for node in nodes]
    value = [(closeness_centrality_met[node]*10)**2 for node in nodes]
    net.add_nodes(nodes,color=color,value=value)
    
    
    maximum_corr = network_data.max()
    for index,value in network_data.items():
        if value>cutoff:
            net.add_edge(index[0],index[1],width=(value**2)*100,color=colorFader(c1,c2,value/maximum_corr)) 

    net.barnes_hut(gravity=-1000, central_gravity=0.1, spring_length=150, spring_strength=0.0015, damping=0.09, overlap=0.8)


    template = net.templateEnv.get_template(net.path)
    net.template = template
    return net

In [13]:
results_env = results[(results.donor!="environment") & (results.receiver!="environment")]
results_env = results_env[["donor","receiver","frequency"]]
# Find mean of frequencies
results_grouped_rec_env = results_env.groupby(["donor","receiver"]).mean()

In [14]:
network_data = results_grouped_rec_env["frequency"].copy()

**Network topological properties**


- **Betweeness_centrality**: Sum of shortest paths that pass through node i
- **Degree centrality**: The fraction of nodes that node i is connected to. 
- **Eigenvector centrality**: Higher if node i is connected to other well connected notes.

In [15]:
centrality_metrics_SP = centrality_metrics(network_data).sort_values("closeness (1/data)")
centrality_metrics_SP

,betweeness (1/data),closeness (1/data)
Eubacterium_rectale_ATCC_33656,0.000000,0.044502
Blautia_hydrogenotrophica_DSM_10507,0.000000,0.046679
Collinsella_aerofaciens_ATCC_25986,0.000000,0.048146
Bacteroides_ovatus_ATCC_8483,0.000000,0.054663
Desulfovibrio_piger_ATCC_29098,0.000000,0.055551
Bacteroides_vulgatus_ATCC_8482,0.000000,0.057392
Bacteroides_uniformis_ATCC_8492,0.022222,0.060329
Bacteroides_thetaiotaomicron_VPI_5482,0.000000,0.062715
Eggerthella_lenta_DSM_2243,0.000000,0.066301
Clostridium_hiranonis_TO_931_DSM_13275,0.244444,0.082585


In [16]:
centrality_metrics(network_data,SP_centrality=False).sort_values("degree (data)")

,degree (data),eigenvector (data)
Blautia_hydrogenotrophica_DSM_10507,0.378983,0.187968
Eubacterium_rectale_ATCC_33656,0.443348,0.202922
Collinsella_aerofaciens_ATCC_25986,0.463363,0.220509
Desulfovibrio_piger_ATCC_29098,0.474848,0.226788
Bacteroides_ovatus_ATCC_8483,0.489151,0.244377
Bacteroides_vulgatus_ATCC_8482,0.584892,0.285765
Bacteroides_thetaiotaomicron_VPI_5482,0.587791,0.305534
Eggerthella_lenta_DSM_2243,0.593487,0.290474
Bacteroides_uniformis_ATCC_8492,0.642653,0.302475
Clostridium_hiranonis_TO_931_DSM_13275,0.902272,0.431696


In [17]:
closeness_centrality_met = centrality_metrics_SP["closeness (1/data)"].to_dict()

In [18]:
net = frequency_network(network_data,closeness_centrality_met=closeness_centrality_met)
net.show("example.html")

example.html


In [19]:
net = frequency_network(network_data,closeness_centrality_met=closeness_centrality_met,cutoff=0.1)
net.show("example.html")

example.html


### Sum mass

In [20]:
def frequency_network(network_data,closeness_centrality_met,cutoff=0):
    net = Network(directed=True)

    nodes = list(set(list(network_data.index.get_level_values(0).unique())+list(network_data.index.get_level_values(1).unique())))
    color = [colorFader(b1,b2,closeness_centrality_met[node]) for node in nodes]
    value = [(closeness_centrality_met[node]*10)**2 for node in nodes]
    net.add_nodes(nodes,color=color,value=value)
    
    
    maximum_corr = network_data.max()
    for index,value in network_data.items():
        if value>cutoff:
            net.add_edge(index[0],index[1],width=(value)*100,color=c2) #colorFader(c1,,value/maximum_corr)

    net.barnes_hut(gravity=-1000, central_gravity=0.1, spring_length=150, spring_strength=0.0015, damping=0.09, overlap=0.8)


    template = net.templateEnv.get_template(net.path)
    net.template = template
    return net

In [21]:
results_env = results[(results.donor!="environment") & (results.receiver!="environment")]
#results_env = results_env[results_env.frequency>0.1]

results_env = results_env[["donor","receiver","mass_rate*frequency"]]
# Find mean of frequencies
results_grouped_rec_env = results_env.groupby(["donor","receiver"]).sum()

In [22]:
network_data = results_grouped_rec_env["mass_rate*frequency"].copy()

**Network topological properties**


- **Betweeness_centrality**: Sum of shortest paths that pass through node i
- **Degree centrality**: The fraction of nodes that node i is connected to. 
- **Eigenvector centrality**: Higher if node i is connected to other well connected notes.

In [23]:
centrality_metrics_SP = centrality_metrics(network_data).sort_values("closeness (1/data)")
centrality_metrics_SP

,betweeness (1/data),closeness (1/data)
Desulfovibrio_piger_ATCC_29098,0.000000,0.000042
Eubacterium_rectale_ATCC_33656,0.000000,0.000085
Blautia_hydrogenotrophica_DSM_10507,0.000000,0.000111
Collinsella_aerofaciens_ATCC_25986,0.000000,0.000174
Eggerthella_lenta_DSM_2243,0.000000,0.000176
Bacteroides_vulgatus_ATCC_8482,0.000000,0.000192
Clostridium_hiranonis_TO_931_DSM_13275,0.355556,0.000229
Bacteroides_uniformis_ATCC_8492,0.088889,0.000233
Bacteroides_ovatus_ATCC_8483,0.111111,0.000234
Bacteroides_thetaiotaomicron_VPI_5482,0.466667,0.000238


In [24]:
centrality_metrics(network_data,SP_centrality=False).sort_values("degree (data)")

PowerIterationFailedConvergence: (PowerIterationFailedConvergence(...), 'power iteration failed to converge within 100 iterations')

In [25]:
closeness_centrality_met = centrality_metrics_SP["closeness (1/data)"].to_dict()

In [26]:
net = frequency_network(network_data,closeness_centrality_met=closeness_centrality_met)
net.show("example.html")

example.html


In [27]:
net = frequency_network(network_data,closeness_centrality_met=closeness_centrality_met,cutoff=1e-3)
net.show("example.html")

example.html


In [28]:
results[(results.receiver=="Clostridium_hiranonis_TO_931_DSM_13275") & (results.donor!="environment") & (results.frequency>0.1)]

,donor,receiver,compound,mass_rate,rate,frequency,community,medium,mass_rate*frequency
77,Bacteroides_thetaiotaomicron_VPI_5482,Clostridium_hiranonis_TO_931_DSM_13275,M_cholate__91__e__93__,7.177007e-03,1.760961e-02,0.227,NONE,LB_extend,1.629181e-03
83,Bacteroides_thetaiotaomicron_VPI_5482,Clostridium_hiranonis_TO_931_DSM_13275,M_C02528__91__e__93__,6.877669e-03,1.756468e-02,0.197,NONE,LB_extend,1.354901e-03
175,Collinsella_aerofaciens_ATCC_25986,Clostridium_hiranonis_TO_931_DSM_13275,M_cholate__91__e__93__,2.353148e-03,5.773718e-03,0.241,NONE,LB_extend,5.671086e-04
225,Eggerthella_lenta_DSM_2243,Clostridium_hiranonis_TO_931_DSM_13275,M_cholate__91__e__93__,1.432827e-03,3.515605e-03,0.413,NONE,LB_extend,5.917574e-04
397,Bacteroides_uniformis_ATCC_8492,Clostridium_hiranonis_TO_931_DSM_13275,M_fru__91__e__93__,5.872432e-04,3.259648e-03,0.426,NONE,LB_extend,2.501656e-04
433,Faecalibacterium_prausnitzii_A2_165,Clostridium_hiranonis_TO_931_DSM_13275,M_fru__91__e__93__,5.156818e-04,2.862428e-03,0.396,NONE,LB_extend,2.042100e-04
779,Eubacterium_rectale_ATCC_33656,Clostridium_hiranonis_TO_931_DSM_13275,M_etoh__91__e__93__,1.312532e-04,2.849107e-03,0.111,NONE,LB_extend,1.456911e-05
1115,Bacteroides_ovatus_ATCC_8483,Clostridium_hiranonis_TO_931_DSM_13275,M_adn__91__e__93__,3.515299e-05,1.315405e-04,0.729,NONE,LB_extend,2.562653e-05
1218,Faecalibacterium_prausnitzii_A2_165,Clostridium_hiranonis_TO_931_DSM_13275,M_ser_L__91__e__93__,2.162380e-05,2.057601e-04,0.116,NONE,LB_extend,2.508361e-06
1499,Eubacterium_rectale_ATCC_33656,Clostridium_hiranonis_TO_931_DSM_13275,M_glu_L__91__e__93__,7.949355e-06,5.440255e-05,0.112,NONE,LB_extend,8.903278e-07


In [128]:
network_data

donor                           receiver                              
Bacteroides_ovatus_ATCC_8483    Bacteroides_thetaiotaomicron_VPI_5482     0.571117
                                Bacteroides_uniformis_ATCC_8492           0.528146
                                Bacteroides_vulgatus_ATCC_8482            0.471814
                                Blautia_hydrogenotrophica_DSM_10507       0.528200
                                Clostridium_hiranonis_TO_931_DSM_13275    0.582841
                                                                            ...   
Prevotella_copri_CB7_DSM_18205  Clostridium_hiranonis_TO_931_DSM_13275    0.474739
                                Collinsella_aerofaciens_ATCC_25986        0.170750
                                Desulfovibrio_piger_ATCC_29098            0.231333
                                Eggerthella_lenta_DSM_2243                0.169300
                                Faecalibacterium_prausnitzii_A2_165       0.156758
Name: frequency,